# Split-MNIST: confirmação, baselines, fairness e ablações

Este notebook reúne todos os baselines na mesma interface. Ele foi entregue **sem executar nenhuma célula**. A confirmação usa uma configuração imutável; baselines, fairness e ablações são análises secundárias e não podem ser usados para reajustar o candidato confirmatório.

In [ ]:
from pathlib import Path
from dataclasses import replace
import json
import pandas as pd

from experiments.confirmatory_split_mnist import (
    CANDIDATE, CONFIRMATORY_SEEDS, FROZEN_CONFIG,
    preregistration_manifest, run_confirmation, validate_preregistration,
)
from experiments.split_mnist_suite import (
    ALL_BASELINES, ABLATION_METHODS, CLASS_ORDERS, baseline_config,
    run_ablation_matrix, run_all_baselines, run_equal_example_budget,
    run_order_and_capacity_generalization,
)
from experiments.visual_generalization import (
    generalization_configs, run_visual_generalization,
)

DATA_DIR = Path('../data')
RESULTS_DIR = Path('../results/split_mnist_protocol')
DEVICE = 'cpu'  # altere apenas o hardware, não o protocolo
RUN_EXPERIMENTS = False

## 1. Pré-registro confirmatório

Endpoint primário: acurácia média final. Diferenças: candidato menos replay, pareadas por seed. O agregado salva IC Student-t, bootstrap pareado e contagens positiva/negativa/empate. O arquivo `preregistration.lock.json` é criado com modo exclusivo antes do primeiro treino, impedindo sobrescrita acidental.

In [ ]:
validate_preregistration()
manifest = preregistration_manifest()
pd.Series({
    'primary_endpoint': manifest['primary_endpoint'],
    'candidate': manifest['candidate'],
    'reference': manifest['reference'],
    'n_seeds': len(manifest['confirmatory_seeds']),
    'sha256': manifest['sha256'],
})

In [ ]:
# Única célula autorizada para a confirmação independente.
if RUN_EXPERIMENTS:
    confirmation = run_confirmation(
        data_dir=DATA_DIR,
        output_dir=RESULTS_DIR / 'confirmation',
        device=DEVICE,
    )

## 2. Todos os baselines em uma execução

A lista contém vanilla, replay, DER++, ER-ACE, A-GEM, EWC, SI, LwF calibrada, replay com loss balanceada, replay com 20 épocas, replay com early stopping e o candidato SlowHeat. Todos compartilham inicialização, batches atuais, memória e índices de replay por seed.

In [ ]:
BASELINE_SEEDS = [311, 617, 919, 1223, 1523]  # secundárias; não confirmatórias
all_methods = pd.DataFrame({'method': ALL_BASELINES})
all_methods

In [ ]:
if RUN_EXPERIMENTS:
    baselines = run_all_baselines(
        seeds=BASELINE_SEEDS,
        data_dir=DATA_DIR,
        output_dir=RESULTS_DIR / 'all_baselines_equal_epochs',
        device=DEVICE,
    )

## 3. Fairness e custo

A execução anterior compara dez épocas (com os dois controles de replay longo explicitamente identificados). A próxima impõe 20.000 exemplos de learner por tarefa, contando atuais + replay. Cada resultado registra tempo total, exemplos atuais/replay, forward do professor, passos, tempo de `optimizer.step`, consolidação, calibração e FLOPs aproximados, com overhead separado para hooks, regularizadores, consolidação e máscaras.

In [ ]:
if RUN_EXPERIMENTS:
    equal_examples = run_equal_example_budget(
        seeds=BASELINE_SEEDS,
        data_dir=DATA_DIR,
        output_dir=RESULTS_DIR / 'all_baselines_equal_examples',
        device=DEVICE,
    )

In [ ]:
def load_cost_table(result_dir):
    frame = pd.read_csv(Path(result_dir) / 'aggregate.csv')
    columns = [
        'method', 'final_average_accuracy_mean', 'elapsed_seconds_mean',
        'learner_examples_processed_mean', 'total_model_examples_processed_mean',
        'estimated_total_flops_mean', 'estimated_overhead_flops_mean',
    ]
    return frame[columns].sort_values('final_average_accuracy_mean', ascending=False)

# load_cost_table(RESULTS_DIR / 'all_baselines_equal_epochs')

## 4. Ablações restantes

Inclui hidden-only sem replay, budget adaptativo, proteção parcial da saída, calibração da cabeça, replay com learning rate global reduzido e memórias de 5/10/20/50/100 exemplos por classe.

In [ ]:
pd.DataFrame({'method': ABLATION_METHODS})

In [ ]:
if RUN_EXPERIMENTS:
    ablations = run_ablation_matrix(
        seeds=BASELINE_SEEDS,
        data_dir=DATA_DIR,
        output_dir=RESULTS_DIR / 'ablations',
        device=DEVICE,
    )

## 5. Generalização

Cinco ordens fixas de classes e MLPs `256-128`, `512-256` e `512-512-256` são executáveis abaixo. A correção usa máscaras de classes vistas, portanto ordens não canônicas mantêm labels globais corretos. Os adapters adicionais preservam cenários distintos: Permuted-MNIST é domain-incremental; Split CIFAR-100 e TinyImageNet são class-incremental. TinyImageNet requer `train/` e `val/` locais em formato ImageFolder e não é baixado automaticamente.

In [ ]:
pd.DataFrame({'order_id': range(len(CLASS_ORDERS)), 'class_order': CLASS_ORDERS})

In [ ]:
if RUN_EXPERIMENTS:
    generalization = run_order_and_capacity_generalization(
        seeds=BASELINE_SEEDS,
        data_dir=DATA_DIR,
        output_dir=RESULTS_DIR / 'split_mnist_generalization',
        device=DEVICE,
    )

In [ ]:
pd.DataFrame([
    {
        'benchmark': name,
        'scenario': config.scenario,
        'tasks': config.task_count,
        'classes': len(config.class_order),
        'hidden_dims': config.hidden_dims,
    }
    for name, config in generalization_configs(DEVICE).items()
] )

In [ ]:
if RUN_EXPERIMENTS:
    permuted = run_visual_generalization(
        'permuted_mnist', seeds=BASELINE_SEEDS, data_dir=DATA_DIR,
        output_dir=RESULTS_DIR / 'permuted_mnist', device=DEVICE,
    )
    cifar100 = run_visual_generalization(
        'split_cifar100', seeds=BASELINE_SEEDS, data_dir=DATA_DIR / 'cifar100',
        output_dir=RESULTS_DIR / 'split_cifar100', device=DEVICE,
    )
    # tiny = run_visual_generalization(
    #     'tiny_imagenet', seeds=BASELINE_SEEDS, data_dir=Path('/dados/tiny-imagenet-200'),
    #     output_dir=RESULTS_DIR / 'tiny_imagenet', device=DEVICE, download=False,
    # )

## 6. Leitura do endpoint primário

Depois da confirmação, leia exclusivamente `paired_differences_vs_replay → slowheat_replay_hidden_beta_30_budget_0.25 → final_average_accuracy → confirmatory` para a decisão primária. Forgetting, gap, custo, outros baselines e outras ordens são secundários.

In [ ]:
def primary_result(aggregate_path):
    with Path(aggregate_path).open(encoding='utf-8') as handle:
        aggregate = json.load(handle)
    return aggregate['paired_differences_vs_replay'][CANDIDATE][
        'final_average_accuracy'
    ]['confirmatory']

# primary_result(RESULTS_DIR / 'confirmation' / 'aggregate.json')